# Fine-Tune Gemma 4 E4B-IT (QLoRA) — 2×T4 Optimized
Larger model (~4.5B effective params). Uses model parallelism across 2 T4s.
**Key:** batch_size=1 + gradient_accumulation=16 to fit in VRAM.
Final GGUF: ~5 GB.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git peft bitsandbytes accelerate datasets huggingface_hub


In [ ]:
import os, sys, torch, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE_DIR = '/kaggle/working'
MODEL_OUT = os.path.join(BASE_DIR, 'gemma_lora_output')
MERGED_DIR = os.path.join(BASE_DIR, 'gemma_merged_fp16')
os.makedirs(MODEL_OUT, exist_ok=True); os.makedirs(MERGED_DIR, exist_ok=True)
model_id = 'google/gemma-4-E4B-it'
n_gpu = torch.cuda.device_count()
for i in range(n_gpu): print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_mem/1e9:.1f} GB')
assert n_gpu >= 2, 'This notebook requires 2 GPUs! Select 2xT4 in Kaggle settings.'


In [ ]:
from huggingface_hub import login
os.environ['WANDB_DISABLED'] = 'true'
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('Logged in via Kaggle Secrets')
except Exception:
    if 'HF_TOKEN' in os.environ: login(token=os.environ['HF_TOKEN'])
    else: print('WARNING: No HF_TOKEN found!')


In [ ]:
from datasets import load_dataset
ds = load_dataset('ajibawa-2023/Children-Stories-Collection', split='train', cache_dir=os.path.join(BASE_DIR, 'hf_cache'))
sample_size = 8000
ds_sample = ds.shuffle(seed=42).select(range(min(sample_size, len(ds))))
PREFIXES = [
    'Level: Age3-4 — Simple words.', 'Level: Age5-6 — Short sentences.',
    'Level: Age7-8 — Moderate vocabulary.', 'Level: Age9-10 — Longer sentences.',
    'Level: Age11-12 — Richer vocabulary.'
]
def format_gemma(ex, idx):
    pfx = PREFIXES[idx % len(PREFIXES)]
    return {'text': f'<start_of_turn>user\n{pfx}\n{ex.get("prompt","")}<end_of_turn>\n<start_of_turn>model\n{ex.get("text","")}<end_of_turn>\n'}
ds_fmt = ds_sample.map(format_gemma, with_indices=True, remove_columns=ds_sample.column_names)
print(f'Formatted {len(ds_fmt)} samples')


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb,
    device_map='balanced', max_memory={0: '14GiB', 1: '14GiB'})
print('Model loaded across GPUs:')
for k, v in model.hf_device_map.items(): print(f'  {k} -> GPU {v}')


In [ ]:
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
for n, p in model.named_parameters():
    p.requires_grad = False
    if p.ndim == 1 and 'norm' in n.lower(): p.data = p.data.to(torch.float32)
try:
    from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
    ct = 0
    for n, m in list(model.named_modules()):
        if isinstance(m, Gemma4ClippableLinear):
            parts = n.split('.')
            setattr(model.get_submodule('.'.join(parts[:-1])), parts[-1], m.linear)
            ct += 1
    print(f'Unwrapped {ct} ClippableLinear modules')
except (ImportError, AttributeError) as e:
    print(f'Unwrap skipped: {e}')
lora_config = LoraConfig(r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
MAX_SEQ_LEN = 512
def tok_fn(ex):
    o = tokenizer(ex['text'], truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    o['labels'] = o['input_ids'].copy()
    return o
ds_tok = ds_fmt.map(tok_fn, batched=True, remove_columns=['text'])
sp = ds_tok.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = sp['train'], sp['test']
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
# batch_size=1 is critical to avoid OOM on E4B with 2xT4
args = TrainingArguments(output_dir=MODEL_OUT, per_device_train_batch_size=1, gradient_accumulation_steps=16,
    optim='paged_adamw_8bit', save_steps=200, save_total_limit=1, logging_steps=20,
    learning_rate=2e-4, max_grad_norm=0.3, num_train_epochs=1, warmup_steps=10,
    lr_scheduler_type='cosine', fp16=True, eval_strategy='steps', eval_steps=200,
    gradient_checkpointing=True)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
print(f'Training with batch_size=1, grad_accum=16 (effective batch=16)')
trainer.train()
trainer.model.save_pretrained(MODEL_OUT); tokenizer.save_pretrained(MODEL_OUT)


In [ ]:
print('Freeing GPU memory...')
del model, trainer
torch.cuda.empty_cache()
gc.collect()


In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
print('Loading base model in fp16 on CPU for merge...')
base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base, MODEL_OUT).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved to {MERGED_DIR}')
del base, merged; gc.collect()


In [ ]:
%%bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp && make -j4


In [ ]:
%%bash
pip install -q -r llama.cpp/requirements.txt
python llama.cpp/convert_hf_to_gguf.py /kaggle/working/gemma_merged_fp16 --outfile /kaggle/working/ft-gemma-e4b-fp16.gguf --outtype f16
rm -rf /kaggle/working/gemma_merged_fp16
./llama.cpp/llama-quantize /kaggle/working/ft-gemma-e4b-fp16.gguf /kaggle/working/ft-gemma-e4b-Q4_K_M.gguf Q4_K_M
rm -f /kaggle/working/ft-gemma-e4b-fp16.gguf
ls -lh /kaggle/working/ft-gemma-e4b-Q4_K_M.gguf


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
GGUF = '/kaggle/working/ft-gemma-e4b-Q4_K_M.gguf'
api.upload_file(path_or_fileobj=GGUF, path_in_repo='finetuned-gemma-4-e4b-it-Q4_K_M.gguf', repo_id='khedim/NLP-MINI-PROJECT',
    commit_message='Upload fine-tuned Gemma 4 E4B Q4_K_M GGUF')
print('Upload Complete!')
